# Anonymous review adaptation

This notebook is adapted from the research implementation. Model labels and output names use **Adjustment**. Personal paths are replaced by a local configuration. Hyperparameters, data screens, and estimation logic are otherwise retained. The MiniLM benchmark uses the same external-snapshot alignment as the other encoders.

**Required:** a locally authorized analysis sample and completed embedding export. These inputs are not bundled. This adaptation has passed static checks and CPU component tests; full neural training has not been rerun in the packaging environment. New code identities create separate checkpoints; original results remain unchanged.


In [ ]:
from pathlib import Path
import runpy
_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p/'src'/'project_config.py').is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or its notebooks folder.")
_config_api = runpy.run_path(str(REPO_ROOT/'src'/'project_config.py'))
REPO_CONFIG = _config_api['load_project_config'](REPO_ROOT)
print('Repository configuration loaded. Input files are read locally; nothing is uploaded.')


# Adjustment benchmarks and asset sensitivity — Llama report embeddings

Same analysis table, financial/year/industry controls, treatment, outcome, architecture, hyperparameters and company folds as the MiniLM benchmark. Only H changes. Run this standalone Windows notebook top to bottom. Requires a **completed** Llama report-vector export; it makes no API calls and performs no embedding inference.

Default: run the full sample (0%). Optional future settings exclude the bottom 1% and 5% by current assets. All eight models refit within each population. The new model needs its own full-sample baseline; MiniLM results are not reused as Llama results. Report vectors are joined by doc_id, never by row position across models. Missing report vectors stop the run rather than changing the sample.

Raw ROA and all earlier provisional causal/timing/identity limitations remain. Sample restrictions change the target population. Results are continuous-treatment residual slopes, not automatically binary ATEs.


## Step 1 — Paths, model selection and export verification
Uses your moved Dropbox project folders. Auto-detects one completed Llama snapshot for the shared corpus; if multiple exports are found, set EMBEDDING_SNAPSHOT explicitly. Matrix/index checksums are verified. No paid API or model download is needed.

Two notebooks may run concurrently: output folders differ. Both neural models use CUDA if available; Ridge/RF use CPU. Concurrent runs share CPU/RAM/GPU and may take longer than sequential runs. Close unnecessary notebooks if memory becomes constrained. Baseline settings are unchanged.


In [ ]:
from pathlib import Path
import os,json,time,hashlib,random,gc
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import joblib
from scipy.stats import t as student_t
from sklearn.model_selection import GroupKFold,GroupShuffleSplit
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset,DataLoader

PROJECT_ROOT=REPO_ROOT
COMPUSTAT_ROOT=REPO_CONFIG['data_root']
EMBEDDING_ROOT=REPO_CONFIG['embedding_root']
DATA_DIR=REPO_CONFIG['analysis_data_dir']
OUTPUT_ROOT=REPO_CONFIG['output_root']/'benchmarks_llama'
assert (DATA_DIR/'dataset_snapshot.json').is_file(), f"Input dataset not found: {DATA_DIR}"
print('Dataset:',DATA_DIR)
print('Embedding folder exists:',EMBEDDING_ROOT.is_dir())
# Report vectors are joined by doc_id using the selected encoder snapshot.
CONTROL_COLS=['w_roa_current','w_log_assets','w_leverage','w_cash_ratio']
CAT_COLS=['fe_year','fe_industry_sic2']
CFG={'implementation_tag':'full_adjustment_v1','seed':42,'n_splits':5,'dZ':64,'encoder_hidden':[256,128],
 'outcome_hidden':[128,64],'adversary_hidden':[64,32],
 'dropout':.1,'grl_lambda':.05,'adv_weight':.005,'lr':.001,
 'weight_decay':1e-5,'epochs':40,'min_epochs':10,'patience':6,
 'batch_size':512,'ridge_alpha':10.,'rf_n_estimators':300,
 'rf_max_depth':12,'rf_min_samples_leaf':20,'rf_max_features':.5,
 'drop_bottom_assets_pct':0,'numeric_controls':CONTROL_COLS,'categorical_controls':CAT_COLS}
CPU_THREADS=min(4,max(1,(os.cpu_count() or 2)-1))
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(CPU_THREADS)
CFG['device']=DEVICE
print('Device:',DEVICE,'| CPU threads:',CPU_THREADS)

from copy import deepcopy
from IPython.display import display
BASE_CFG=deepcopy(CFG)
BASELINE_RUN_DIR=None  # Auto-detect the completed baseline in OUTPUT_ROOT.
SENSITIVITY_PCTS=[0]  # Proposal baseline; set [0,1,5] for planned sensitivity analyses.
MAKE_FIGURES=True
SENSITIVITY_ROOT=OUTPUT_ROOT/'asset_sensitivity'
SENSITIVITY_ROOT.mkdir(parents=True,exist_ok=True)

EMBEDDING_ENGINE='llama'
EXPECTED_CORPUS_ID=REPO_CONFIG['expected_corpus_id']
EMBEDDING_SNAPSHOT=REPO_CONFIG['embedding_snapshots'].get('llama')

def embedding_sha(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
 return h.hexdigest()

def resolve_embedding():
 if EMBEDDING_SNAPSHOT is not None:
  choices=[Path(EMBEDDING_SNAPSHOT)]
 else:
  choices=[]
  for path in (EMBEDDING_ROOT/EMBEDDING_ENGINE/EXPECTED_CORPUS_ID).rglob('latest_snapshot.json'):
   candidate=json.loads(path.read_text(encoding='utf-8'))
   if candidate.get('complete') is True and candidate.get('corpus_id')==EXPECTED_CORPUS_ID:
    choices.append(path)
 if len(choices)!=1:
  raise RuntimeError(f'Expected one completed {EMBEDDING_ENGINE} snapshot; found {len(choices)}: {choices}. Finish export or set EMBEDDING_SNAPSHOT explicitly.')
 path=choices[0];snapshot=json.loads(path.read_text(encoding='utf-8'))
 assert snapshot.get('complete') is True, 'Embedding export is incomplete.'
 assert snapshot['corpus_id']==EXPECTED_CORPUS_ID, 'Different text corpus.'
 matrix=path.parent/snapshot['report_matrix'];index=path.parent/snapshot['report_index']
 assert embedding_sha(matrix)==snapshot['report_matrix_sha256'], 'Embedding matrix checksum mismatch.'
 assert embedding_sha(index)==snapshot['report_index_sha256'], 'Embedding index checksum mismatch.'
 return path,snapshot,matrix,index

_embedding_cache={}
def aligned_report_vectors(source):
 if 'matrix' not in _embedding_cache:
  idx=pd.read_csv(SELECTED_INDEX,dtype={'doc_id':'string','corpus_id':'string'})
  matrix=np.load(SELECTED_MATRIX,mmap_mode='r',allow_pickle=False)
  assert idx.doc_id.notna().all() and idx.doc_id.is_unique, 'Missing/duplicate embedding doc_id.'
  rows=pd.to_numeric(idx.embedding_row,errors='raise').to_numpy(dtype=float)
  assert np.isfinite(rows).all() and np.equal(rows,np.floor(rows)).all()
  assert np.array_equal(np.sort(rows),np.arange(len(idx))), 'Invalid embedding row index.'
  assert matrix.shape==(len(idx),int(SELECTED_SNAPSHOT['dimension']))
  assert len(idx)==int(SELECTED_SNAPSHOT['reports_exported'])
  assert idx.corpus_id.eq(EXPECTED_CORPUS_ID).all()
  _embedding_cache.update(index=idx.set_index('doc_id'),matrix=matrix)
 idx=_embedding_cache['index'];matrix=_embedding_cache['matrix']
 missing=source.loc[~source.doc_id.isin(idx.index),'doc_id']
 assert missing.empty, f'Missing {len(missing)} report embeddings. No silent sample deletion is allowed.'
 matched=idx.loc[source.doc_id]
 assert np.allclose(pd.to_numeric(matched.competition_intensity),pd.to_numeric(source.competition_intensity),rtol=1e-8,atol=1e-10), 'Competition text corpus mismatch.'
 aligned=np.asarray(matrix[matched.embedding_row.to_numpy(dtype=np.int64)],dtype=np.float32)
 assert np.isfinite(aligned).all() and (np.linalg.norm(aligned,axis=1)>0).all()
 return aligned

SELECTED_SNAPSHOT_PATH,SELECTED_SNAPSHOT,SELECTED_MATRIX,SELECTED_INDEX=resolve_embedding()
print('Verified embedding snapshot:',SELECTED_SNAPSHOT_PATH)
print('Model:',SELECTED_SNAPSHOT['settings']['model'],'| Dimension:',SELECTED_SNAPSHOT['dimension'])


## Step 2 — Unchanged neural and nuisance functions
Architecture, early stopping, Ridge/RF settings and financial/year/industry controls are identical to the baseline. NoAdv and Adjustment both feed Ridge and RF. Company folds are assigned on the original sample first, then restricted; retained companies keep their baseline test fold.

In [ ]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, gradient):
        return -ctx.lambd * gradient, None

def mlp(in_dim, hidden, out_dim, dropout):
    layers = []
    for width in hidden:
        layers += [nn.Linear(in_dim, width), nn.ReLU(), nn.Dropout(dropout)]
        in_dim = width
    return nn.Sequential(*layers, nn.Linear(in_dim, out_dim))

class AdjustmentContinuous(nn.Module):
    def __init__(self, dH, dW, cfg):
        super().__init__()
        dZ, drop = cfg["dZ"], cfg["dropout"]
        self.encoder = mlp(dH, cfg["encoder_hidden"], dZ, drop)
        self.y_head = mlp(dZ + dW + 1, cfg["outcome_hidden"], 1, drop)
        self.t_adv = mlp(dZ, cfg["adversary_hidden"], 1, drop)

def tensor(x):
    return torch.as_tensor(x, dtype=torch.float32, device=DEVICE)

@torch.no_grad()
def encode_batches(model, h, batch_size=2048):
    model.eval()
    return np.concatenate([
        model.encoder(tensor(h[i:i + batch_size])).cpu().numpy()
        for i in range(0, len(h), batch_size)
    ])

def fit_representation(htr, wtr, ttr, ytr, hva, wva, tva, yva, seed, cfg):
    set_seed(seed)
    model = AdjustmentContinuous(htr.shape[1], wtr.shape[1], cfg).to(DEVICE)
    main_params = list(model.encoder.parameters()) + list(model.y_head.parameters())
    adv_params = list(model.t_adv.parameters())
    main_opt = torch.optim.AdamW(main_params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    adv_opt = torch.optim.AdamW(adv_params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    train_data = TensorDataset(*[torch.tensor(a, dtype=torch.float32)
                                for a in [htr, wtr, ttr, ytr]])
    loader = DataLoader(train_data, batch_size=cfg["batch_size"],
                        shuffle=True, num_workers=0)
    hv, wv, tv, yv = [tensor(a) for a in [hva, wva, tva, yva]]
    best, best_state, best_epoch, bad = float("inf"), None, None, 0
    history = []
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        train_y_sum, train_adv_sum, n_train = 0.0, 0.0, 0
        for hb, wb, tb, yb in loader:
            hb, wb, tb, yb = [a.to(DEVICE) for a in [hb, wb, tb, yb]]
            # Main update: reverse the gradient going to Z, freeze adversary weights.
            for p in adv_params:
                p.requires_grad_(False)
            main_opt.zero_grad(set_to_none=True)
            z = model.encoder(hb)
            yhat = model.y_head(torch.cat([z, wb, tb[:, None]], dim=1)).squeeze(-1)
            that = model.t_adv(GradReverse.apply(z, cfg["grl_lambda"])).squeeze(-1)
            ly, la = F.mse_loss(yhat, yb), F.mse_loss(that, tb)
            (ly + cfg["adv_weight"] * la).backward()
            nn.utils.clip_grad_norm_(main_params, 5.0)
            main_opt.step()
            # Adversary update: no gradient to encoder.
            for p in adv_params:
                p.requires_grad_(True)
            adv_opt.zero_grad(set_to_none=True)
            with torch.no_grad():
                detached_z = model.encoder(hb)
            adv_loss = F.mse_loss(model.t_adv(detached_z).squeeze(-1), tb)
            adv_loss.backward()
            nn.utils.clip_grad_norm_(adv_params, 5.0)
            adv_opt.step()
            train_y_sum += ly.item() * len(hb)
            train_adv_sum += la.item() * len(hb)
            n_train += len(hb)
        model.eval()
        with torch.no_grad():
            zv = model.encoder(hv)
            yp = model.y_head(torch.cat([zv, wv, tv[:, None]], dim=1)).squeeze(-1)
            tp = model.t_adv(zv).squeeze(-1)
            val_y = F.mse_loss(yp, yv).item()
            val_adv = F.mse_loss(tp, tv).item()
        assert np.isfinite(val_y) and np.isfinite(val_adv), "Nonfinite training loss."
        history.append({"epoch": epoch, "train_y_mse": train_y_sum / n_train,
            "train_adv_mse": train_adv_sum / n_train,
            "val_y_mse": val_y, "val_adv_mse": val_adv})
        # Only checkpoints after the minimum warm-up are eligible.
        if epoch >= cfg["min_epochs"]:
            if val_y < best - 1e-6:
                best, best_epoch, bad = val_y, epoch, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= cfg["patience"]:
                    break
    assert best_state is not None, "epochs must be >= min_epochs"
    model.load_state_dict(best_state)
    model.eval()
    return model, pd.DataFrame(history), best_epoch

def r2_np(actual, predicted):
    denom = np.sum((actual - np.mean(actual)) ** 2)
    return float(1 - np.sum((actual - predicted) ** 2) / denom) if denom > 1e-12 else np.nan

def make_learner(kind, seed):
    if kind == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=CFG["ridge_alpha"], solver="svd"))
    if kind == "RF":
        return RandomForestRegressor(
            n_estimators=CFG["rf_n_estimators"], max_depth=CFG["rf_max_depth"],
            min_samples_leaf=CFG["rf_min_samples_leaf"],
            random_state=seed, n_jobs=CPU_THREADS, max_features=CFG["rf_max_features"],
        )
    raise ValueError(kind)

def nuisance_predictions(xtr, xte, ttr, ytr, learner, seed):
    treatment_model = make_learner(learner, seed)
    outcome_model = make_learner(learner, seed + 1000)
    treatment_model.fit(xtr, ttr)
    outcome_model.fit(xtr, ytr)
    return treatment_model.predict(xte), outcome_model.predict(xte)

def dml_summary(t, y, that, yhat, cluster_ids):
    u, v = t - that, y - yhat
    denom = float(np.mean(u ** 2))
    if denom < 1e-12:
        raise ValueError("Treatment residual variation is essentially zero.")
    theta = float(np.mean(u * v) / denom)
    influence = u * (v - theta * u) / denom
    sums = pd.DataFrame({"g": cluster_ids, "psi": influence}).groupby("g")["psi"].sum()
    n, G = len(t), len(sums)
    assert G > 1
    se = float(np.sqrt((G / (G - 1)) * np.sum(sums.to_numpy() ** 2) / n ** 2))
    critical = student_t.ppf(0.975, df=G - 1)
    p = float(2 * student_t.sf(abs(theta / se), df=G - 1)) if se > 0 else np.nan
    return {
        "N": n, "companies": G,
        "slope_roa_decimal_per_T_pp": theta,
        "effect_roa_pp_per_T_pp": 100 * theta,
        "se_roa_pp": 100 * se,
        "ci_low_roa_pp": 100 * (theta - critical * se),
        "ci_high_roa_pp": 100 * (theta + critical * se),
        "p_value_approx": p,
        "R2_T_oof": r2_np(t, that), "R2_Y_oof": r2_np(y, yhat),
        "t_resid_mse": denom,
        "residual_mse_to_T_variance": denom / float(np.var(t)),
        "low_abs_residual_share": float(np.mean(np.abs(u) < 0.05 * np.std(t))),
    }

def atomic_npz(path, **arrays):
    temp = path.with_name(path.name + ".tmp")
    with temp.open("wb") as f:
        np.savez_compressed(f, **arrays)
    os.replace(temp, path)

# Confirm gradient reversal sign/scaling.
p=torch.tensor([2.],requires_grad=True)
GradReverse.apply(p,.05).sum().backward()
assert torch.allclose(p.grad,torch.tensor([-.05]))

def get_representation(view,fold,tr,te):
 path=RUN_DIR/f'{view}_representation_fold{fold}.npz'
 if path.is_file():
  with np.load(path,allow_pickle=False) as s:
   assert np.array_equal(s['train_idx'],tr) and np.array_equal(s['test_idx'],te)
   return s['Z_train'].copy(),s['Z_test'].copy()
 split=GroupShuffleSplit(n_splits=1,test_size=.2,random_state=CFG['seed']+fold)
 it,iv=next(split.split(H[tr],groups=groups[tr]));itr,iva=tr[it],tr[iv]
 assert set(groups[itr]).isdisjoint(groups[iva])
 hsc=StandardScaler().fit(H[itr]);wprep=fit_w(itr)
 tsc=StandardScaler().fit(T[itr,None]);ysc=StandardScaler().fit(Y[itr,None])
 cfg=dict(CFG)
 if view=='NoAdv':cfg.update(adv_weight=0.,grl_lambda=0.)
 model,history,epoch=fit_representation(
  hsc.transform(H[itr]),apply_w(wprep,itr),tsc.transform(T[itr,None]).ravel(),ysc.transform(Y[itr,None]).ravel(),
  hsc.transform(H[iva]),apply_w(wprep,iva),tsc.transform(T[iva,None]).ravel(),ysc.transform(Y[iva,None]).ravel(),
  CFG['seed']+fold,cfg)
 zsc=StandardScaler().fit(encode_batches(model,hsc.transform(H[itr])))
 ztr=zsc.transform(encode_batches(model,hsc.transform(H[tr]))).astype('float32')
 zte=zsc.transform(encode_batches(model,hsc.transform(H[te]))).astype('float32')
 history.to_csv(RUN_DIR/f'{view}_training_fold{fold}.csv',index=False)
 joblib.dump({'H':hsc,'W':wprep,'T':tsc,'Y':ysc,'Z':zsc},RUN_DIR/f'{view}_preprocessors_fold{fold}.joblib')
 torch.save({'state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()},'config':cfg,'selected_epoch':epoch},RUN_DIR/f'{view}_encoder_fold{fold}.pt')
 atomic_npz(path,train_idx=tr,test_idx=te,Z_train=ztr,Z_test=zte)
 print(f'{view} fold {fold}: selected epoch {epoch}',flush=True)
 del model;gc.collect()
 if torch.cuda.is_available():torch.cuda.empty_cache()
 return ztr,zte


## Step 3 — Processing programs
These strings contain the same verified input loading, eight-model estimation and final figures as the original notebook. The loading program now preserves baseline folds across the two restrictions. Settings and the code digest create separate checkpoint folders.

In [ ]:
CODE_DIGEST='5a30409c46082b6c87719bdb01feaeca848baded921371cd02e763fe20600a21'
DATA_PROGRAM="def file_sha(path):\n h=hashlib.sha256()\n with Path(path).open('rb') as f:\n  for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)\n return h.hexdigest()\n\nsnap=json.loads((DATA_DIR/'dataset_snapshot.json').read_text(encoding='utf-8'))\nassert snap['complete']\ntable_path=DATA_DIR/snap['table'];matrix_path=DATA_DIR/snap['matrix']\nassert file_sha(table_path)==snap['table_sha256']\n# The new embedding matrix/index checksums were verified in Step 1.\nsource=pd.read_csv(table_path,dtype={'doc_id':'string','gvkey':'string',**{c:'string' for c in CAT_COLS}},low_memory=False)\nsource_H=aligned_report_vectors(source)\nsnap=dict(snap)\nsnap['matrix_sha256']=SELECTED_SNAPSHOT['report_matrix_sha256']\nassert len(source)==len(source_H) and source.doc_id.is_unique\nassert np.array_equal(source.embedding_row.to_numpy(),np.arange(len(source)))\nassert not source.duplicated(['gvkey','comp_record_id']).any()\ncutoff=source.w_log_assets.quantile(CFG['drop_bottom_assets_pct']/100) if CFG['drop_bottom_assets_pct'] else -np.inf\nkeep=source.w_log_assets.ge(cutoff).to_numpy()\ndf=source.loc[keep].reset_index(drop=True).copy();H=source_H[keep].astype('float32')\nT=(100*pd.to_numeric(df.competition_intensity)).to_numpy(dtype='float64')\nY=pd.to_numeric(df.roa_next_fyear).to_numpy(dtype='float64')\nWnum=df[CONTROL_COLS].to_numpy(dtype='float64')\nWcat=df[CAT_COLS].astype(str).to_numpy()\nassert df[CAT_COLS].notna().all().all() and df.gvkey.notna().all()\nassert all(np.isfinite(x).all() for x in [H,T,Y,Wnum])\nassert np.var(T)>0 and np.var(Y)>0\ngroups=df.gvkey.astype(str).to_numpy()\n# Preserve baseline company folds before applying either asset restriction.\nsource_groups=source.gvkey.astype(str).to_numpy()\nunique_source=np.unique(source_groups)\nshuffled=np.random.default_rng(CFG['seed']).permutation(unique_source)\nlabel_map={g:i for i,g in enumerate(shuffled)}\nsource_labels=np.array([label_map[g] for g in source_groups])\nsource_fold=np.zeros(len(source),dtype=int)\nfor k,(_,te0) in enumerate(GroupKFold(n_splits=CFG['n_splits']).split(source_H,groups=source_labels),1):\n source_fold[te0]=k\nfold_labels=source_fold[keep]\nFOLDS=[(np.flatnonzero(fold_labels!=k),np.flatnonzero(fold_labels==k)) for k in range(1,CFG['n_splits']+1)]\nunique=np.unique(groups)\nfor tr,te in FOLDS:\n assert len(tr)>0 and len(te)>0\n assert set(groups[tr]).isdisjoint(groups[te])\nversions={k:metadata.version(k) for k in ['numpy','pandas','torch','scipy','scikit-learn','joblib']}\nrun_key={'config':CFG,'versions':versions,'code':CODE_DIGEST,'embedding':{'engine':EMBEDDING_ENGINE,'corpus_id':EXPECTED_CORPUS_ID,'index_sha256':SELECTED_SNAPSHOT['report_index_sha256'],'settings':SELECTED_SNAPSHOT['settings']},'table_sha':snap['table_sha256'],'matrix_sha':snap['matrix_sha256']}\nRUN_ID=hashlib.sha256(json.dumps(run_key,sort_keys=True).encode()).hexdigest()[:20]\nRUN_DIR=OUTPUT_ROOT/RUN_ID;RUN_DIR.mkdir(parents=True,exist_ok=True)\n(RUN_DIR/'run_config.json').write_text(json.dumps({**run_key,'N':len(df),'companies':len(unique),'asset_log_cutoff':None if not np.isfinite(cutoff) else cutoff,'status':'provisional_continuous_treatment_analysis','fold_policy':'baseline_company_folds_then_subset'},indent=2),encoding='utf-8')\ndf[['doc_id','gvkey','comp_record_id']].assign(outer_fold=fold_labels).to_csv(RUN_DIR/'sample_folds.csv',index=False)\nprint('Reports:',len(df),'| Companies:',len(unique),'| Output:',RUN_DIR)\n\ndef set_seed(seed):\n random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n\ndef fit_w(indices):\n scaler=StandardScaler().fit(Wnum[indices])\n encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False,dtype=np.float32).fit(Wcat[indices])\n return scaler,encoder\n\ndef apply_w(prep,indices):\n scaler,encoder=prep\n return np.column_stack([scaler.transform(Wnum[indices]),encoder.transform(Wcat[indices])]).astype('float32')\nsource[['doc_id']].assign(embedding_row_in_model=_embedding_cache['index'].loc[source.doc_id].embedding_row.to_numpy()).to_csv(RUN_DIR/'report_embedding_alignment.csv',index=False)\n"
MODEL_PROGRAM="def run_benchmarks():\n specs=[(view,learner) for view in ['W_only','Raw_H','NoAdv','Adjustment'] for learner in ['Ridge','RF']]\n predictions={f'{v}-{l}':{'T_hat':np.full(len(df),np.nan),'Y_hat':np.full(len(df),np.nan)} for v,l in specs}\n probe_names=['T_from_H','T_from_Z','T_from_HW','T_from_ZW','Y_from_HWT','Y_from_ZWT']\n probes={name:np.full(len(df),np.nan) for name in probe_names}\n for fold,(tr,te) in enumerate(FOLDS,1):\n  wprep=fit_w(tr);wtr,wte=apply_w(wprep,tr),apply_w(wprep,te)\n  hprep=StandardScaler().fit(H[tr]);htr,hte=hprep.transform(H[tr]),hprep.transform(H[te])\n  views={'W_only':(wtr,wte),'Raw_H':(np.column_stack([htr,wtr]),np.column_stack([hte,wte]))}\n  # The same learned representation feeds both nuisance learners.\n  for view in ['NoAdv','Adjustment']:\n   ztr,zte=get_representation(view,fold,tr,te)\n   views[view]=(np.column_stack([ztr,wtr]),np.column_stack([zte,wte]))\n  for view,learner in specs:\n   name=f'{view}-{learner}';path=RUN_DIR/f'{name}_predictions_fold{fold}.npz'\n   if path.is_file():\n    with np.load(path,allow_pickle=False) as s:\n     assert np.array_equal(s['test_idx'],te);tp=s['T_hat'].copy();yp=s['Y_hat'].copy()\n   else:\n    print(f'Fold {fold}: fit {name}',flush=True)\n    tp,yp=nuisance_predictions(*views[view],T[tr],Y[tr],learner,CFG['seed']+fold)\n    atomic_npz(path,test_idx=te,T_hat=tp,Y_hat=yp)\n   assert np.isfinite(tp).all() and np.isfinite(yp).all()\n   predictions[name]['T_hat'][te]=tp;predictions[name]['Y_hat'][te]=yp\n  # Independent out-of-fold probes; never pool Z coordinates from different encoders.\n  ztr,zte=get_representation('Adjustment',fold,tr,te)\n  probe_views={\n   'T_from_H':(htr,hte,T),'T_from_Z':(ztr,zte,T),\n   'T_from_HW':(*views['Raw_H'],T),'T_from_ZW':(*views['Adjustment'],T),\n   'Y_from_HWT':(np.column_stack([views['Raw_H'][0],T[tr]]),np.column_stack([views['Raw_H'][1],T[te]]),Y),\n   'Y_from_ZWT':(np.column_stack([views['Adjustment'][0],T[tr]]),np.column_stack([views['Adjustment'][1],T[te]]),Y)}\n  probe_path=RUN_DIR/f'probes_fold{fold}.npz'\n  if probe_path.is_file():\n   with np.load(probe_path,allow_pickle=False) as s:\n    assert np.array_equal(s['test_idx'],te)\n    fold_probes={name:s[name].copy() for name in probe_names}\n  else:\n   fold_probes={}\n   for name,(xtr,xte,target) in probe_views.items():\n    mdl=make_learner('Ridge',CFG['seed']+fold);mdl.fit(xtr,target[tr]);fold_probes[name]=mdl.predict(xte)\n   atomic_npz(probe_path,test_idx=te,**fold_probes)\n  for name in probe_names:probes[name][te]=fold_probes[name]\n  print(f'Fold {fold}/{len(FOLDS)} COMPLETE',flush=True)\n rows=[]\n for name,p in predictions.items():\n  assert all(np.isfinite(x).all() for x in p.values())\n  row=dml_summary(T,Y,p['T_hat'],p['Y_hat'],groups)\n  row.update(model=name,status='provisional_continuous_treatment_slope')\n  rows.append(row)\n result=pd.DataFrame(rows);result.to_csv(RUN_DIR/'benchmark_results.csv',index=False)\n probe_table=pd.DataFrame([{'probe':name,'R2_oof':r2_np(T if name.startswith('T_') else Y,pred)} for name,pred in probes.items()])\n probe_table.to_csv(RUN_DIR/'representation_diagnostics.csv',index=False)\n oof=df[['doc_id','gvkey','comp_record_id','fe_year','fe_industry_sic2']].copy()\n oof['outer_fold']=fold_labels;oof['T_pp']=T;oof['Y_roa']=Y\n for name,p in predictions.items():\n  oof[name+'__T_hat']=p['T_hat'];oof[name+'__Y_hat']=p['Y_hat']\n  oof[name+'__T_residual']=T-p['T_hat'];oof[name+'__Y_residual']=Y-p['Y_hat']\n for name,pred in probes.items():oof[name]=pred\n oof.to_csv(RUN_DIR/'oof_predictions.csv.gz',index=False,compression='gzip')\n # Exact slope recalculation omitting a company's fixed OOF scores (not full retraining).\n influence_rows=[]\n for name,p in predictions.items():\n  u=T-p['T_hat'];v=Y-p['Y_hat']\n  contribution=pd.DataFrame({'gvkey':groups,'a':u*v,'b':u*u}).groupby('gvkey')[['a','b']].sum()\n  A,B=contribution.sum();full=A/B\n  for gv,r in contribution.iterrows():\n   denom=B-r['b']\n   value=(A-r['a'])/denom if denom>0 else np.nan\n   influence_rows.append({'model':name,'omitted_gvkey':gv,'full_effect_roa_pp':100*full,\n     'effect_without_company_fixed_oof':100*value,'change_roa_pp':100*(value-full)})\n influence=pd.DataFrame(influence_rows);influence['absolute_change']=influence.change_roa_pp.abs()\n influence.sort_values('absolute_change',ascending=False).to_csv(RUN_DIR/'company_influence_fixed_oof.csv',index=False)\n (RUN_DIR/'complete.json').write_text(json.dumps({'complete':True,'run_id':RUN_ID,'models':len(result),'reports':len(df),'companies':len(np.unique(groups))},indent=2))\n return result,probe_table,predictions,probes\n\nstarted=time.perf_counter()\nresults,diagnostics,predictions,probes=run_benchmarks()\nprint('Elapsed minutes:',round((time.perf_counter()-started)/60,1))\ndisplay(results)\ndisplay(diagnostics)\nprint('Saved:',RUN_DIR)\n"
FIGURE_PROGRAM="# Diagnostics use held-out predictions; plots never alter the estimation sample.\nfig,axes=plt.subplots(2,2,figsize=(11,8))\nfor row,target,prefix in [(0,T,'T'),(1,Y,'Y')]:\n names=['T_from_H','T_from_Z'] if prefix=='T' else ['Y_from_HWT','Y_from_ZWT']\n for col,name in enumerate(names):\n  pred=probes[name];ax=axes[row,col]\n  b=pd.DataFrame({'actual':target,'predicted':pred})\n  b['bin']=pd.qcut(b.predicted,20,duplicates='drop')\n  means=b.groupby('bin',observed=True)[['predicted','actual']].mean()\n  ax.scatter(means.predicted,means.actual,color='#147D92')\n  low=min(means.min());high=max(means.max());ax.plot([low,high],[low,high],'--',color='gray')\n  ax.set(title=f'{name} | OOF R2={r2_np(target,pred):.3f}',xlabel='Mean predicted value per bin',ylabel='Mean observed value per bin')\nfig.suptitle('Held-out representation probes: binned calibration (descriptive)')\nfig.tight_layout();fig.savefig(RUN_DIR/'figure1_oof_probes.png',dpi=180);plt.show()\n\n# Plot a single encoder fold. Fit the projection on training companies only.\nVISUAL_FOLD=1\nUSE_UMAP=True  # If umap-learn is unavailable, PCA is used and labeled explicitly.\ntr,te=FOLDS[VISUAL_FOLD-1]\nztr,zte=get_representation('Adjustment',VISUAL_FOLD,tr,te)\nrng=np.random.default_rng(CFG['seed'])\nitr=rng.choice(len(tr),min(3000,len(tr)),replace=False)\nite=rng.choice(len(te),min(3000,len(te)),replace=False)\nprojector_class=None\nif USE_UMAP:\n try:\n  import umap\n  projector_class=umap.UMAP\n except ImportError:\n  print('umap-learn is not installed; using PCA for figure 2.')\nfig,axes=plt.subplots(1,2,figsize=(11,4.5))\nfor ax,a,b,title in zip(axes,[H[tr],ztr],[H[te],zte],['Raw H','Adjustment Z']):\n scaler=StandardScaler().fit(a)\n atr=scaler.transform(a[itr]);ate=scaler.transform(b[ite])\n if projector_class is not None:\n  projection=projector_class(n_components=2,n_neighbors=30,min_dist=.3,random_state=42,n_jobs=1).fit(atr)\n  coordinates=projection.transform(ate);method='UMAP'\n else:\n  projection=PCA(n_components=2,random_state=42).fit(atr)\n  coordinates=projection.transform(ate);method='PCA'\n color=ax.scatter(coordinates[:,0],coordinates[:,1],c=T[te][ite],cmap='viridis',s=7,alpha=.6,rasterized=True,vmin=T[te][ite].min(),vmax=T[te][ite].max())\n ax.set(title=f'{title}: held-out fold {VISUAL_FOLD}',xlabel=method+' 1',ylabel=method+' 2')\n fig.colorbar(color,ax=ax,label='Competition intensity (pp)')\nfig.suptitle('Same held-out reports; separate projections; exploratory visualization')\nfig.tight_layout();fig.savefig(RUN_DIR/'figure2_H_Z_treatment.png',dpi=180);plt.show()\n\nfig,axes=plt.subplots(1,2,figsize=(11,4))\nfor ax,learner in zip(axes,['Ridge','RF']):\n for view in ['W_only','Raw_H','NoAdv','Adjustment']:\n  name=f'{view}-{learner}';residual=T-predictions[name]['T_hat']\n  ax.hist(residual,bins=100,density=True,histtype='step',label=view)\n ax.set(title=learner+' OOF treatment residuals',xlabel='T minus predicted T (pp)',ylabel='Density')\n ax.legend()\nfig.tight_layout();fig.savefig(RUN_DIR/'figure3_treatment_residuals.png',dpi=180);plt.show()\nprint('Figures saved:',RUN_DIR)\n"

## Step 4 — Fit selected populations
Long-running cell. Each population refits all eight models with separate checkpoints. Restart and run Steps 1–4 to resume completed folds. A partial neural fold restarts. Do not start two writers for the same embedding version. Same company test folds across all populations and embedding versions.


In [ ]:
BASELINE_RUN_DIR=None  # Fit this embedding's own baseline at exclusion_pct=0.
sensitivity_runs=[]
for pct in SENSITIVITY_PCTS:
 print('\nCURRENT-ASSET EXCLUSION:',pct,'%',flush=True)
 CFG=deepcopy(BASE_CFG)
 CFG['drop_bottom_assets_pct']=pct
 CFG['implementation_tag']='full_adjustment_asset_sensitivity_v1'
 OUTPUT_ROOT=SENSITIVITY_ROOT
 exec(compile(DATA_PROGRAM,'sensitivity_data','exec'),globals())
 # Quantiles are computed on the original full eligible sample, never on Y.
 manifest=pd.DataFrame({'doc_id':source.doc_id,'gvkey':source.gvkey,
  'current_log_assets':source.w_log_assets,'retained':keep})
 manifest.to_csv(RUN_DIR/'asset_sample_selection.csv.gz',index=False,compression='gzip')
 exec(compile(MODEL_PROGRAM,'sensitivity_models','exec'),globals())
 sensitivity_runs.append({'exclusion_pct':pct,'folder':str(RUN_DIR)})
 if MAKE_FIGURES:
  exec(compile(FIGURE_PROGRAM,'sensitivity_figures','exec'),globals())
  plt.close('all')
 if torch.cuda.is_available():torch.cuda.empty_cache()
 gc.collect()
print('All three populations finished for',EMBEDDING_ENGINE)


## Step 5 — Compare all three populations
The full-sample reference comes from this embedding's own 0% run. Check effect sizes, confidence intervals, nuisance/probe diagnostics and company influence; do not pick a sample by significance. Fixed-OOF company influence is not a full deletion-and-refit exercise. Figures are stored in each run folder.


In [ ]:
runs=([{'exclusion_pct':0,'folder':str(BASELINE_RUN_DIR)}] if BASELINE_RUN_DIR is not None else [])+sensitivity_runs
all_results=[];all_probes=[];all_influence=[]
for entry in runs:
 folder=Path(entry['folder']);pct=entry['exclusion_pct']
 result=pd.read_csv(folder/'benchmark_results.csv')
 assert len(result)==8 and result.model.is_unique
 result.insert(0,'excluded_bottom_assets_pct',pct)
 result['run_folder']=str(folder)
 all_results.append(result)
 probe=pd.read_csv(folder/'representation_diagnostics.csv')
 probe.insert(0,'excluded_bottom_assets_pct',pct);all_probes.append(probe)
 inf=pd.read_csv(folder/'company_influence_fixed_oof.csv',dtype={'omitted_gvkey':'string'})
 top=inf.sort_values('absolute_change',ascending=False).groupby('model',sort=False).head(1).copy()
 top.insert(0,'excluded_bottom_assets_pct',pct);all_influence.append(top)
comparison=pd.concat(all_results,ignore_index=True)
probe_comparison=pd.concat(all_probes,ignore_index=True)
influence_comparison=pd.concat(all_influence,ignore_index=True)
comparison_id=hashlib.sha256(json.dumps(runs,sort_keys=True).encode()).hexdigest()[:16]
COMPARISON_DIR=SENSITIVITY_ROOT/('comparison_'+comparison_id)
COMPARISON_DIR.mkdir(parents=True,exist_ok=True)
comparison.to_csv(COMPARISON_DIR/'benchmark_comparison.csv',index=False)
probe_comparison.to_csv(COMPARISON_DIR/'probe_comparison.csv',index=False)
influence_comparison.to_csv(COMPARISON_DIR/'largest_company_influence_comparison.csv',index=False)
(COMPARISON_DIR/'runs.json').write_text(json.dumps(runs,indent=2),encoding='utf-8')
cols=['excluded_bottom_assets_pct','N','companies','model','effect_roa_pp_per_T_pp',
 'ci_low_roa_pp','ci_high_roa_pp','p_value_approx','R2_T_oof','R2_Y_oof']
display(comparison[cols].round(4))
display(probe_comparison)
display(influence_comparison)
print('SEND BACK THESE THREE COMPARISON CSV FILES:',COMPARISON_DIR)


## Interpretation
Compare baseline, 1% and 5% without picking a preferred specification from its p-value. Check whether company influence declines, the estimated magnitude becomes plausible, results remain sensitive across learners, and Adjustment improves over the same-budget NoAdv benchmark. A smaller p-value does not by itself establish a better estimator. Assess causal assumptions separately before HTE interpretation.

Individual run folders include the same three diagnostic figures as before. No new embedding, SEC download or Compustat matching is needed.
